<a href="https://colab.research.google.com/github/MarkChernov123/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [15]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])

# TODO: assert the row count and the quantity total
lines = pd.json_normalize(orders_json,record_path=['lines'],meta=['order', 'vendor_id'])

print(lines)
print("\nTotal Rows:", len(lines))
print("Sum of Quantities:", lines['qty'].sum())


           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07

Total Rows: 4
Sum of Quantities: 7


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
#had difficulty attaching a left merge even after reviewing lecture and studio, used ai as a reference to guide some of the code.


vendors_df = pd.DataFrame(vendors_json)
joined = pd.merge(
    lines, vendors_df, on='vendor_id', how='left', indicator=True
)
unmatched_df = joined[joined['_merge'] == 'left_only']

missing_vendor_id = unmatched_df['vendor_id'].iloc[0]
total_units = unmatched_df['qty'].sum()

print(joined)
print(f'\nVendor that did not match: {missing_vendor_id}')
print(f'Total units attached to it: {total_units}')

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only

Vendor that did not match: V-07
Total units attached to it: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO

#unmatched vendor is listed as NaN from the left join as seen above,
#to prevent it from getting dropped when getting grouped, I set the
#dropna to False so that the NaN vendor's stats still get summed

vendor_totals_df = (
    joined.groupby('name', dropna=False)['qty'].sum().reset_index(name='total_qty'))

print(vendor_totals_df)


           name  total_qty
0     Cav Merch          1
1  Hoos Burgers          3
2           NaN          3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [6]:
# TODO

zone_totals_df = (
    joined.groupby('zone', dropna=False)['qty'].sum().reset_index(name='total_qty'))

print(zone_totals_df)
print(f"Total sum across zones: {zone_totals_df['total_qty'].sum()}")

  zone  total_qty
0    A          4
1  NaN          3
Total sum across zones: 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [13]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
#generates the rows using dictionary method
rows = []
for order_dict in ragged_json:
    order_id = order_dict.get('order')
    vendor_id = order_dict.get('vendor_id')
    order_lines = order_dict.get('lines')

    # looks through orders that have no lines, like order 5
    if not order_lines:
        rows.append({
            'order': order_id,
            'vendor_id': vendor_id,
            'item': None,
            'qty': None
        })
    else:
        # handles the orders that do have the lines category
        for order_line in order_lines:
            rows.append({
                'order': order_id,
                'vendor_id': vendor_id,
                'item': order_line.get('item'),
                'qty': order_line.get('qty', None)
            })
ragged_df = pd.DataFrame(rows)

print(ragged_df)

   order vendor_id   item  qty
0      4      V-01   Soda  2.0
1      5      V-10   None  NaN
2      6      V-01  Fries  NaN


### Q6 — Validate

In [16]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

_your answer here_

A)

It makes it easier to work with each transaction individually

B)
Treating missing data as a zero can alter averages of totals. replacing NaN with a 0 may create errors in fields that require strings and can create errors in that manner when transporting data from one table to another.